# Phi-4-mini LoRA for text-to-SQL (CSE 572, Phase D)

Free Kaggle GPU. **Settings (right panel):** Accelerator = **GPU T4 x2**, Internet = **on**, and add the dataset `phi4mini-sft` (re-upload it if `kaggle_train.py` changed).

Run with **Save Version → Save & Run All (Commit)**; it keeps running with the browser closed. Outputs land in `/kaggle/working/<run>/`; download the `.gguf` files from the version's *Output* tab.

* `base` (D0 control) was produced by the first version: download it from there, no need to rerun.
* `RUNS = "d1,d2"` trains both adapters back to back. Each stops cleanly after `--max-hours 4.5`, so both fit in Kaggle's 12-hour session. The log prints a projected training time at step 5.

In [ ]:
!pip install -q -U "transformers>=4.56" "peft>=0.17" "bitsandbytes>=0.46" accelerate sentencepiece protobuf
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob, shutil
src = glob.glob('/kaggle/input/**/kaggle_train.py', recursive=True)
assert src, 'kaggle_train.py not found: attach the phi4mini-sft dataset to this notebook'
shutil.copy(src[0], '/kaggle/working/kaggle_train.py')
print('data folders:', sorted(glob.glob('/kaggle/input/**/train.jsonl', recursive=True)))

In [ ]:
RUNS = "d1,d2"
!cd /kaggle/working && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python kaggle_train.py --runs {RUNS} --steps 400 --max-hours 4.5

In [ ]:
!ls -lh /kaggle/working/*/
!rm -f /kaggle/working/kaggle_train.py
import json, glob
for f in glob.glob('/kaggle/working/*/train_log.json'):
    l = json.load(open(f)); print(f, 'val loss:', [round(v['loss'], 3) for v in l['val']], 'stopped early:', l.get('stopped_early_at_step'))